```{contents}
```
## Backpropagation 


**Backpropagation** is the algorithm that enables neural networks to learn.
It efficiently computes how each parameter (weight and bias) should be adjusted to minimize a loss function by applying the **chain rule of calculus** through the network layers.

Training loop structure:

| Phase            | Purpose             |
| ---------------- | ------------------- |
| Forward Pass     | Compute predictions |
| Loss Computation | Measure error       |
| Backward Pass    | Compute gradients   |
| Parameter Update | Apply optimization  |

Without backpropagation, deep networks would be computationally infeasible to train.

---

### Mathematical Foundation

Given:

* Model: $\hat{y} = f(x; \theta)$
* Loss: $\mathcal{L}(\hat{y}, y)$
* Objective: minimize $\mathcal{L}$ w.r.t parameters $\theta$

Backprop computes:

$$
\frac{\partial \mathcal{L}}{\partial \theta}
$$

by recursively applying the **chain rule** from output layer to input layer:

$$
\frac{\partial \mathcal{L}}{\partial w} =
\frac{\partial \mathcal{L}}{\partial z_L}
\cdot
\frac{\partial z_L}{\partial z_{L-1}}
\cdots
\frac{\partial z_1}{\partial w}
$$

---

### Why It Works Efficiently

Naively computing gradients separately for each parameter costs exponential time.
Backprop reuses intermediate derivatives, reducing complexity to **O(number of parameters)**.

This is equivalent to **reverse-mode automatic differentiation**.

---

### Backpropagation Workflow

1. **Forward pass**

   * Compute activations layer by layer
2. **Compute loss**
3. **Backward pass**

   * Start from loss gradient
   * Propagate gradients backward using chain rule
4. **Parameter update**

   * Use optimizer (SGD, Adam, etc.)

---

### Visual Flow

```
Input → [Layer1] → [Layer2] → ... → [Output]
           ↑           ↑             ↑
        ∂L/∂W1     ∂L/∂W2         ∂L/∂WL
```

Gradients flow in the opposite direction of data.

---

### PyTorch Demonstration

#### Simple Neural Network with Backprop



In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Dataset
x = torch.randn(100, 2)
y = (x[:, 0] + x[:, 1] > 0).float().unsqueeze(1)

# Model
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid()
)

# Loss & Optimizer
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training Loop
for epoch in range(50):
    optimizer.zero_grad()         # 1. Clear old gradients
    y_pred = model(x)             # 2. Forward pass
    loss = loss_fn(y_pred, y)     # 3. Compute loss
    loss.backward()               # 4. Backpropagation
    optimizer.step()              # 5. Update parameters

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 0.6755
Epoch 10, Loss: 0.6136
Epoch 20, Loss: 0.5589
Epoch 30, Loss: 0.4991
Epoch 40, Loss: 0.4363




#### Inspecting Gradients



In [2]:
for name, param in model.named_parameters():
    print(name, param.grad.norm())


0.weight tensor(0.1444)
0.bias tensor(0.0657)
2.weight tensor(0.1759)
2.bias tensor(0.0301)




---

### Autograd Mechanism in PyTorch

PyTorch builds a **dynamic computation graph** during the forward pass.

* Each tensor stores:

  * `data`
  * `grad`
  * `grad_fn` (how it was computed)

Calling `loss.backward()` triggers reverse traversal of this graph.

---

### Variants and Enhancements

| Variant                | Purpose                                    |
| ---------------------- | ------------------------------------------ |
| Standard Backprop      | General supervised learning                |
| Stochastic Backprop    | Mini-batch training                        |
| Truncated BPTT         | RNN training on long sequences             |
| Second-Order Backprop  | Uses Hessians (rare in practice)           |
| Gradient Checkpointing | Saves memory by recomputing forward states |

---

### Common Practical Issues

| Problem             | Cause                       | Solution                   |
| ------------------- | --------------------------- | -------------------------- |
| Vanishing gradients | Deep networks, sigmoid/tanh | ReLU, BatchNorm, Residuals |
| Exploding gradients | Large weights               | Gradient clipping          |
| Slow convergence    | Poor initialization         | He/Xavier initialization   |
| Overfitting         | Excessive capacity          | Dropout, regularization    |

---

### Summary

| Property      | Description                       |
| ------------- | --------------------------------- |
| Purpose       | Compute gradients efficiently     |
| Core Tool     | Chain rule                        |
| Direction     | Output → Input                    |
| Complexity    | Linear in number of parameters    |
| Foundation of | All modern deep learning training |

Backpropagation is the mathematical engine that makes deep learning trainable at scale.

### Why We Rarely Customize `backward()`

**Because PyTorch’s `autograd` already generates the exact symbolic gradients for any differentiable computation graph, automatically, efficiently, and correctly.**

Manually writing `backward()` is:

* error-prone
* hard to maintain
* slower to prototype
* unnecessary for most research and production workloads